# Chapter 15: Algorithm Complexity

Explore Big O by counting operations, compare time and extra memory, and distinguish verification from exhaustive search. Includes runnable worked solutions for all seven supplied exercises.

Select **JavaScript (Node.js)** and run all cells from the top. No external packages are needed. Restart the kernel before rerunning all cells if declarations already exist. Counts describe the stated operation, not elapsed milliseconds. We assume fixed-size JavaScript numbers and constant-time comparisons unless stated otherwise.

This notebook helps you answer: **If my input gets larger, how much more work and memory will my program need?** Instead of timing a program on one computer, it counts meaningful operations. You will move from one increment, to scanning a collection, to nested loops, to halving a search range, and finally to trying combinations.

Each code cell prints a table or example result and uses `assert` to check the expected answer. An assertion stays silent when it passes and throws an error if the result is wrong. `console.table` arranges results into columns; `console.log` prints ordinary values. Run the cells in order because later examples reuse earlier functions.


**If a name is not defined:** its defining cell has not run in the current kernel, or the name was typed with different capitalization. Choose **Kernel → Restart Kernel and Run All Cells** to recreate all definitions in order. The function is named `sequentialSearch`, not `sequentialsearch`.


In [ ]:
const assert = require('node:assert/strict');

## 1. Big O and constant work

Big O is an asymptotic upper bound on resource growth with input size. Time and space are separate resources; best and worst cases are separate analyses. A tight bound describes the actual growth class. Constant factors and lower-order terms disappear: 3n + 10 is O(n).

| Growth | Name | Example work |
| --- | --- | --- |
| O(1) | Constant | One numeric increment |
| O(log n) | Logarithmic | Repeatedly halve a search range |
| O((log n)ᶜ) | Polylogarithmic | A fixed power of a logarithm |
| O(n) | Linear | Scan every input item |
| O(n log n) | Linearithmic | Merge sort |
| O(n²) | Quadratic | Compare all pairs |
| O(nᶜ) | Polynomial | Fixed nonnegative power c |
| O(cⁿ) | Exponential | Enumerate subsets when c = 2 |

### Exercise 1: Count constant work
Predict increment(1), increment(2), and increment(1000). Which changes: the answer or the number of increments?

**What the next cell does:** `increment` adds one to its numeric argument and returns the new value. Prefix `++num` increments before returning. The three calls produce 2, 3, and 1001. The `increments` column is a stated count of the one increment in the function; it is not a timer or an automatically collected measurement.

Here, input value and input size are different ideas: a larger fixed-size Number does not make the function loop more times. For array examples later, n means the number of elements. O(1) means the amount of work stays bounded as the relevant input size grows, rather than that execution takes exactly one CPU instruction.


In [ ]:
function increment(num) {
  return ++num;
}
const incrementRows = [1, 2, 1000].map(input => ({
  input, result: increment(input), increments: 1
}));
console.table(incrementRows);
assert.deepEqual(incrementRows.map(row => row.result), [2, 3, 1001]);
assert.ok(incrementRows.every(row => row.increments === 1));

The return value changes; each call still performs one increment: O(1) time and O(1) extra space.

## 2. Linear work: sequential search

Count calls to equalsFn. Return the measurement alongside the index so successful and unsuccessful searches both report their cost. The comparator must itself cost O(1) for the stated bound.

### Exercise 2: Trace a sequential search
For [4, 7, 9, 12, 15, 21], predict the index and comparison count for targets 4, 12, 21, and 20.

**How the next cell works:** Start at index 0, compare the current item with the target, and stop as soon as they match. `equalsFn` lets the caller define what a match means; its default uses strict equality (`===`). Increment `comparisons` immediately before each call so the matching item is included in the count.

The returned object contains `index` (the matching position, or −1 if absent) and `comparisons` (items inspected). Searching for 12 examines 4, 7, 9, then 12, so the result is index 3 after four comparisons. The last table deliberately searches for missing values in arrays of 10 and 1000 items: every item must be checked. Sequential search is useful when data is unsorted or the collection is small.


In [ ]:
function sequentialSearch(array, value, equalsFn = (a, b) => a === b) {
  let comparisons = 0;
  for (let i = 0; i < array.length; i++) {
    comparisons++;
    if (equalsFn(value, array[i])) return { index: i, comparisons };
  }
  return { index: -1, comparisons };
}
const searchValues = [4, 7, 9, 12, 15, 21];
const searchRows = [4, 12, 21, 20].map(target => ({
  target, ...sequentialSearch(searchValues, target, (a, b) => a === b)
}));
console.table(searchRows);
assert.deepEqual(searchRows.map(({ index, comparisons }) => [index, comparisons]),
  [[0, 1], [3, 4], [5, 6], [-1, 6]]);
console.table([10, 1000].map(n => ({ n,
  ...sequentialSearch(Array.from({ length: n }, (_, i) => i + 1), n + 1)
})));

For n ≥ 1, the best case makes one comparison and the worst makes n. Finding the last item and missing the target do the same work. An empty array makes zero comparisons. Worst-case time is O(n), extra space O(1).

## 3. Quadratic work: bubble sort

This version completes n passes with n − 1 comparisons each. It mutates the input. Its cost counter includes each outer pass and each comparison: n + n(n − 1) = n². Comparisons alone total n(n − 1).

### Exercise 3: Count nested loops
Trace [3, 1, 2], then predict the counts at n = 6. Does sorted input change these counts? Compare two consecutive n-step loops with an n-step loop containing three constant-time operations.

**How the next cell works:** Bubble sort compares neighboring items and swaps them if they are out of order. On [3, 1, 2], the first pass changes [3, 1, 2] to [1, 3, 2], then [1, 2, 3]. The remaining passes still run even though the data is already sorted.

`compareFn(a, b)` returns a positive number when a should come after b. Destructuring assignment swaps the two array entries. The returned `array` is the same array supplied by the caller, now sorted; this is what mutating the input means. `passes` counts outer-loop iterations, `comparisons` counts neighbor comparisons, and `cost` adds those two counts. It does not count swaps or every JavaScript instruction.

`countLoopPatterns` then counts two other arrangements of work. Two loops placed one after another add their costs. A nested loop multiplies iteration counts, but a fixed inner length of 3 still gives only 3n operations. Bubble sort is easy to trace, though more efficient sorting algorithms suit large collections.


In [ ]:
function bubbleSort(array, compareFn = (a, b) => a - b) {
  let passes = 0;
  let comparisons = 0;
  for (let i = 0; i < array.length; i++) {
    passes++;
    for (let j = 0; j < array.length - 1; j++) {
      comparisons++;
      if (compareFn(array[j], array[j + 1]) > 0) {
        [array[j], array[j + 1]] = [array[j + 1], array[j]];
      }
    }
  }
  return { array, passes, comparisons, cost: passes + comparisons };
}
const bubbleThree = bubbleSort([3, 1, 2]);
const bubbleSix = bubbleSort([6, 5, 4, 3, 2, 1]);
console.log(bubbleThree, bubbleSix);
assert.deepEqual(bubbleThree, { array: [1, 2, 3], passes: 3, comparisons: 6, cost: 9 });
assert.equal(bubbleSix.passes, 6);
assert.equal(bubbleSix.comparisons, 30);
assert.equal(bubbleSix.cost, 36);
assert.equal(bubbleSort([1, 2, 3]).comparisons, 6);
function countLoopPatterns(n) {
  let consecutive = 0, nested = 0;
  for (let i = 0; i < n; i++) consecutive++;
  for (let i = 0; i < n; i++) consecutive++;
  for (let i = 0; i < n; i++) {
    for (let j = 0; j < 3; j++) nested++;
  }
  return { consecutive, nested };
}
assert.deepEqual(countLoopPatterns(6), { consecutive: 12, nested: 18 });
console.log(countLoopPatterns(6));

Sorted input does not change the comparison count here: even the best case is quadratic. An early-exit bubble sort can achieve O(n) best-case time. The two loop patterns perform 2n and 3n operations respectively; both are O(n). Nesting alone does not imply quadratic growth.

## 4. Compare growth rates

The table uses base-10 logarithms to match the supplied chapter. Logarithm bases differ by a constant factor, so they have the same Big O class. Values are illustrative functions, not measured runtimes. Exponential values use BigInt to avoid numeric overflow.

**How to read the next table:** Each row uses the same n across several growth functions. Compare columns within a row to see how different strategies scale, then compare rows to see what increasing input size does. At n = 1000, the illustrative linear cost is 1000 and the quadratic cost is 1,000,000. These values are neither seconds nor predictions of exact runtime.

`Math.log10` calculates a base-10 logarithm. `toFixed(2)` rounds the display to two decimal places. `2n` is JavaScript BigInt syntax for the integer two; it does not mean two times the input size. `2n ** BigInt(n)` computes an exact power of two. Very long results are displayed as their first eight digits followed by their total digit count. Computing this one power does not enumerate that many subsets.


In [ ]:
const growthRows = [10, 20, 50, 100, 500, 1000, 10000].map(n => {
  const exponential = 2n ** BigInt(n);
  const digits = exponential.toString();
  return { n, constant: 1, log10: Number(Math.log10(n).toFixed(2)),
    linear: n, nLog10: Number((n * Math.log10(n)).toFixed(2)),
    quadratic: n * n,
    exponential: digits.length <= 16 ? digits : `${digits.slice(0, 8)}… (${digits.length} digits)` };
});
console.table(growthRows);

### Exercise 4: Double the input

From 10 to 20, linear work grows by 2, quadratic by 4, and 2ⁿ by 1024. Exactly n² comparisons grow from 1,000,000 at n = 1000 to 4,000,000 at n = 2000.

**Why the ratios differ:** Linear work changes from n to 2n, so its ratio is 2. Quadratic work changes from n² to (2n)² = 4n². Exponential work changes from 2ⁿ to 2²ⁿ, so the ratio is 2ⁿ and depends on the original n. The ratio 1024 applies specifically to doubling from 10 to 20; it is not a constant doubling rule for exponential growth.


In [ ]:
assert.equal(20 / 10, 2);
assert.equal(20 ** 2 / 10 ** 2, 4);
assert.equal(2 ** 20 / 2 ** 10, 1024);
assert.equal(2000 ** 2, 4000000);
console.log('Doubling ratios:', { linear: 2, quadratic: 4, exponentialAt10: 1024 });

## 5. Logarithmic work: binary search

Binary search requires sorted input. This implementation uses the lower middle index. A comparison means one inspection of a middle value through the comparator, even though the code may check its result more than once.

### Exercise 5: Trace a binary search
Search for 16 in [2, 4, 6, 8, 10, 12, 14, 16]. Record inspected values and remaining candidates after each miss.

**How the next cell works:** `low` and `high` mark the inclusive range of possible positions. Inspect the middle item. If it is smaller than the target, discard it and everything to its left; otherwise, if it is larger, discard it and everything to its right. Stop when the target is found or `low > high` means no candidates remain. The comparator must match the array's sort order.

For target 16, the first inspected value is 8. The target is larger, leaving [10, 12, 14, 16]. Inspecting 12 leaves [14, 16]; inspecting 14 leaves [16]; the next inspection succeeds. The trace's `remaining` column counts candidates after a miss. On the successful row it shows the range size at discovery, rather than a range left to search.

The returned `index` identifies the match, `comparisons` counts middle-item inspections, and `trace` explains the decisions. On a nonempty array the worst-case count for this implementation is floor(log₂ n) + 1. Binary search is useful for repeated lookups in data that is already sorted; maintaining that sorted order has a separate cost.


The next cell includes its own `sequentialSearch` definition for the linear-search comparison. JavaScript names are case-sensitive: use `sequentialSearch` with a capital S in Search. Run the initial assertion setup cell before running examples individually.


In [ ]:
// Include the linear baseline here so this comparison does not require Exercise 2.
function sequentialSearch(array, value, equalsFn = (a, b) => a === b) {
  let comparisons = 0;
  for (let i = 0; i < array.length; i++) {
    comparisons++;
    if (equalsFn(value, array[i])) return { index: i, comparisons };
  }
  return { index: -1, comparisons };
}

function binarySearchTrace(array, value, compareFn = (a, b) => a - b) {
  let low = 0, high = array.length - 1;
  const trace = [];
  while (low <= high) {
    const mid = low + Math.floor((high - low) / 2);
    const comparison = compareFn(array[mid], value);
    if (comparison === 0) {
      trace.push({ inspected: array[mid], found: true, remaining: high - low + 1 });
      return { index: mid, comparisons: trace.length, trace };
    }
    if (comparison < 0) low = mid + 1;
    else high = mid - 1;
    trace.push({ inspected: array[mid], found: false, remaining: Math.max(0, high - low + 1) });
  }
  return { index: -1, comparisons: trace.length, trace };
}
const binaryValues = [2, 4, 6, 8, 10, 12, 14, 16];
const binaryResult = binarySearchTrace(binaryValues, 16);
console.table(binaryResult.trace);
assert.deepEqual(binaryResult.trace.map(step => step.inspected), [8, 12, 14, 16]);
assert.deepEqual(binaryResult.trace.filter(step => !step.found).map(step => step.remaining), [4, 2, 1]);
assert.equal(binaryResult.index, 7);
assert.equal(sequentialSearch(binaryValues, 16).comparisons, 8);
const binaryWorstComparisons = n => n === 0 ? 0 : Math.floor(Math.log2(n)) + 1;
assert.equal(binaryWorstComparisons(8), 4);
assert.equal(binaryWorstComparisons(16), 5);
console.log('Worst-case comparisons at sizes 8 and 16:', binaryWorstComparisons(8), binaryWorstComparisons(16));

Doubling the sorted array from 8 to 16 raises the worst-case count from 4 to 5. Iterative binary search uses O(1) extra space; this teaching version records O(log n) trace entries. Sorting an unsorted input first adds its own time and space costs.

## 6. Time and extra memory

### Exercise 6: Compare two scans
Method A stores the maximum so far. Method B also copies each value. Both take O(n) time, assuming amortized O(1) array append. A uses O(1) extra space; B uses O(n), including its returned copy. Input storage is excluded. These functions return undefined as the maximum of an empty array.

**How the next cell works:** Both functions inspect every value and update `maximum` whenever a larger value appears. Starting with `undefined` allows the first value to become the maximum, including when every value is negative. The examples assume ordinary numeric inputs without NaN.

`maximumScan` returns just 12 for [3, −2, 12, 7]. `maximumWithCopy` returns an object with `maximum: 12` and a new `copy` containing all four values in their original order. Neither function sorts or changes the input. The copy is a separate array: its growing storage explains the O(n) extra-space bound. A single numeric maximum and a few loop variables need only O(1) extra space. Equal time complexity therefore does not imply equal memory use.


In [ ]:
function maximumScan(array) {
  let maximum;
  for (const value of array) {
    if (maximum === undefined || value > maximum) maximum = value;
  }
  return maximum;
}
function maximumWithCopy(array) {
  let maximum;
  const copy = [];
  for (const value of array) {
    if (maximum === undefined || value > maximum) maximum = value;
    copy.push(value);
  }
  return { maximum, copy };
}
const memoryValues = [3, -2, 12, 7];
assert.equal(maximumScan(memoryValues), 12);
assert.deepEqual(maximumWithCopy(memoryValues), { maximum: 12, copy: memoryValues });
assert.notEqual(maximumWithCopy(memoryValues).copy, memoryValues);
console.log(maximumScan(memoryValues), maximumWithCopy(memoryValues));

## 7. Complexity reference: assumptions matter

These are conventional bounds under the stated implementations. JavaScript engine behavior and representation can affect actual performance.

| Structure / operation | Typical or average | Worst / qualification |
| --- | --- | --- |
| Array index access | O(1) | Assumes direct access |
| Array append / stack push | O(1) amortized | A resize can take O(n) |
| Stack pop / indexed queue dequeue | O(1) | Queue uses a head index; array shift is O(n) |
| Array / list search | O(n) | O(n) |
| Linked list insertion / removal at known node | O(1) | Finding the node or predecessor can cost O(n) |
| Hash lookup / insert / delete | O(1) expected | O(n) with collisions; resizing adds cost |
| Unbalanced BST search / insert / delete | O(log n) for suitably distributed inputs | O(n) |
| AVL / red-black tree search / insert / delete | O(log n) | O(log n) |
| Binary heap peek / insert / extract | O(1) / O(log n) / O(log n) | Arbitrary search is O(n); capacity changes may add cost |

For graphs, V is the vertex set and E the edge set. Matrix bounds assume a dense, dynamically resized matrix; list bounds assume direct access to a vertex's neighbor list.

| Graph operation | Adjacency list | Adjacency matrix |
| --- | --- | --- |
| Storage | O(\|V\| + \|E\|) | O(\|V\|²) |
| Add vertex | O(1) amortized | O(\|V\|²) if copying the matrix |
| Add edge | O(1) amortized without duplicate check | O(1) |
| Remove vertex | O(\|V\| + \|E\|) | O(\|V\|²) if rebuilding |
| Remove / query edge (u, v) | O(deg(u)) | O(1) |
| DFS / BFS | O(\|V\| + \|E\|) | O(\|V\|²) |

| Sort | Best time | Average time | Worst time |
| --- | --- | --- | --- |
| Bubble, full passes used above | O(n²) | O(n²) | O(n²) |
| Bubble with early exit | O(n) | O(n²) | O(n²) |
| Selection | O(n²) | O(n²) | O(n²) |
| Insertion | O(n) | O(n²) | O(n²) |
| Shell | Depends on gaps | Depends on gaps | O(n²) for original halving gaps |
| Merge | O(n log n) | O(n log n) | O(n log n) |
| Quick | O(n log n) | O(n log n) expected with suitable pivots | O(n²) |
| Heap | O(n log n) | O(n log n) | O(n log n) |
| Counting | O(n + k) | O(n + k) | O(n + k), k is key range |
| Bucket | O(n + k) | O(n + k) under uniform distribution | O(n² + k) with insertion-sorted buckets |
| Radix | O(d(n + b)) | O(d(n + b)) | O(d(n + b)), d digits, base b |

Sequential search has O(n) worst-case time. Binary search has O(log n) worst-case time on a sorted array. Interpolation search has O(n) worst-case time; its O(log log n) expected bound needs suitable uniformly distributed numeric keys. DFS and BFS bounds assume constant-time visited checks and efficient adjacency access.

**How to use these tables:** First name the exact operation and the representation. “Delete from a linked list” could mean unlinking a node you already have, or searching for a value and then unlinking it; the second includes the search cost. A heap exposes its smallest or largest item quickly, but finding an arbitrary value still requires a search.

“Average” or “expected” depends on assumptions about inputs or randomization. “Amortized” spreads occasional expensive operations over a sequence: an array resize may copy many elements, while appending many items still averages constant work per append. “Worst” bounds the most expensive allowed case. These meanings are different.

For graph tables, deg(u) is the number of neighbors of vertex u. An adjacency list stores actual connections; a matrix reserves a position for every possible pair. Lists usually suit sparse graphs, while matrices provide direct edge queries at the cost of quadratic storage. For sorting tables, read n as item count and check the meanings of k, d, and b before comparing bounds.


## 8. P, NP, NP-complete, and NP-hard

These classes concern decision problems (yes/no questions) and the bit length of their encoded input.

- **P:** solvable in polynomial time.
- **NP:** yes instances have certificates verifiable in polynomial time. P is a subset of NP; whether P = NP remains an open question.
- **Polynomial-time reduction L → M:** transform an instance of L to an instance of M with the same answer. An efficient solver for M would then solve L efficiently.
- **NP-hard:** every problem in NP reduces to this problem in polynomial time.
- **NP-complete:** both in NP and NP-hard. SAT, decision subset sum, decision traveling salesman (tour length at most a bound), and decision vertex cover are examples. Optimization traveling salesman is NP-hard.

NP-complete does not mean impossible or undecidable: exhaustive search can solve finite instances. The halting problem is undecidable and is also NP-hard under polynomial-time reductions. No polynomial-time algorithm for an NP-complete problem is known.

### Exercise 7: Verify a proposed subset
Given [3, 7, 10, 12], target 19, and zero-based indices [1, 3], check valid bounds and distinctness before summing. There are 2ⁿ subsets including the empty subset. Verification checks one certificate; exhaustive search may explore every subset.

**What the next cell demonstrates:** Subset sum asks whether some selection of the input items adds to a target. Items are identified by position, so repeated numeric values at different positions are still different items. A proposed list of positions is a certificate: information that a verifier can check.

`verifySubsetSum` rejects noninteger, out-of-range, or repeated indices. A `Set` records positions already used. It then adds the selected values and returns true only if the sum equals the target. For [1, 3], it selects 7 and 12, giving 19. Returning false means that particular proposal failed; another selection might still work.

`exhaustiveSubsetSum` tries excluding and including each item through recursion. At a leaf, every item has a decision, so one complete subset has been formed. It returns a matching index array, or null after all possibilities fail. The empty array is a valid solution for target zero, which is why the code checks explicitly for null. `visitedLeaves` counts complete subsets examined, not every recursive call. The search stops on its first match and does not promise the smallest subset.


In [ ]:
function verifySubsetSum(values, target, indices) {
  const used = new Set();
  let sum = 0;
  for (const index of indices) {
    if (!Number.isInteger(index) || index < 0 || index >= values.length || used.has(index)) {
      return false;
    }
    used.add(index);
    sum += values[index];
  }
  return sum === target;
}
// Educational exhaustive search: stop at the first solution; include the empty subset.
function exhaustiveSubsetSum(values, target) {
  const indices = [];
  let visitedLeaves = 0;
  function visit(index, sum) {
    if (index === values.length) {
      visitedLeaves++;
      return sum === target ? indices.slice() : null;
    }
    const excluded = visit(index + 1, sum);
    if (excluded !== null) return excluded;
    indices.push(index);
    const included = visit(index + 1, sum + values[index]);
    indices.pop();
    return included;
  }
  const solution = visit(0, 0);
  return { indices: solution, visitedLeaves };
}
const subsetValues = [3, 7, 10, 12];
assert.equal(verifySubsetSum(subsetValues, 19, [1, 3]), true);
console.log('Proposed values:', [subsetValues[1], subsetValues[3]], 'sum:', 7 + 12);
const subsetResult = exhaustiveSubsetSum(subsetValues, 19);
assert.ok(verifySubsetSum(subsetValues, 19, subsetResult.indices));
const impossibleSubset = exhaustiveSubsetSum(subsetValues, 100);
assert.equal(impossibleSubset.indices, null);
assert.equal(impossibleSubset.visitedLeaves, 2 ** subsetValues.length);
console.log('Solution:', subsetResult, 'Unsuccessful exhaustive search:', impossibleSubset);

For k proposed indices, this verifier takes expected O(k) time and O(k) space using a Set, assuming constant-time numeric arithmetic and expected constant-time Set operations. For a valid distinct certificate k ≤ n. A Boolean marker array instead gives O(n + k) initialization and verification time and O(n) space. Arbitrary-precision integer arithmetic also depends on the number of bits.

The recursive search explores O(2ⁿ) nodes in the worst case, with O(n) recursion and working-index space (plus an O(n) returned solution). Keeping a running sum avoids rescanning every subset. Large instances are deliberately excluded from these examples.

## 9. Heuristics and practice

Heuristics such as local search and genetic algorithms can find useful candidates for difficult optimization problems without guaranteeing an optimal result. An approximation algorithm, by contrast, comes with a proved quality bound under specified assumptions. Neither makes an undecidable problem generally solvable.

For practice, pick a small search or sorting task, identify its input size, predict the operation count, and check the prediction. The supplied chapter suggests platforms including Project Euler, HackerRank, SPOJ, Coderbyte, CodeChef, Topcoder, and UVa. Platform offerings and contest details are outside this notebook's scope.

## 10. Verification

Check empty inputs, comparator behavior, successful and failed searches, exact operation-count formulas, and invalid subset certificates. All seven worked exercises also contain assertions above.

**Reading the subset-search output:** The proposed [1, 3] certificate passes. The exhaustive search also finds [1, 3], after examining six complete subsets in its exclude-first order. Searching for target 100 fails after all 16 subsets of four items are checked. With one more input item, each existing subset has two possibilities—leave the new item out or include it—doubling the total.

This difference is the reason verification and discovery are separate questions. Being given a route, assignment, or subset can make checking easy even when finding one is expensive. The small example illustrates that difference; it is not a proof that every subset-sum instance requires exhaustive search.

**Try changing an example:** Search for 4 instead of 16 in the binary-search cell, reverse the initial bubble-sort array, or propose [1, 1] to the subset verifier. Predict the result and the count first. If you change an input used in a worked solution, update its expected assertions as well.

**What the final checks do:** They exercise empty collections, missing targets, sorted results, reversed comparator order, operation-count formulas, and invalid subset indices. They use small deterministic inputs to catch mistakes without running an expensive benchmark. A passing check confirms the tested behavior; complexity conclusions come from analyzing the loops and recursion.


In [ ]:
assert.deepEqual(sequentialSearch([], 1), { index: -1, comparisons: 0 });
assert.equal(sequentialSearch([{ id: 2 }], { id: 2 }, (a, b) => a.id === b.id).index, 0);
assert.deepEqual(bubbleSort([]), { array: [], passes: 0, comparisons: 0, cost: 0 });
for (let n = 0; n <= 20; n++) {
  const values = Array.from({ length: n }, (_, i) => n - i);
  const result = bubbleSort(values);
  assert.equal(result.array, values);
  assert.deepEqual(result.array, Array.from({ length: n }, (_, i) => i + 1));
  assert.equal(result.passes, n);
  assert.equal(result.comparisons, n === 0 ? 0 : n * (n - 1));
  assert.equal(result.cost, n * n);
  const sorted = Array.from({ length: n }, (_, i) => i * 2);
  for (let target = -1; target <= n * 2; target++) {
    const search = binarySearchTrace(sorted, target);
    assert.equal(search.index, sorted.indexOf(target));
    assert.ok(search.comparisons <= binaryWorstComparisons(n));
  }
}
assert.deepEqual(bubbleSort([1, 3, 2], (a, b) => b - a).array, [3, 2, 1]);
assert.equal(maximumScan([]), undefined);
assert.deepEqual(maximumWithCopy([]), { maximum: undefined, copy: [] });
assert.equal(maximumScan([-9, -3, -7]), -3);
for (const indices of [[1, 1], [-1], [4], [1.5], ['1'], [NaN]]) {
  assert.equal(verifySubsetSum(subsetValues, 19, indices), false);
}
assert.equal(verifySubsetSum(subsetValues, 20, [1, 3]), false);
assert.equal(verifySubsetSum([], 0, []), true);
assert.deepEqual(exhaustiveSubsetSum([], 0), { indices: [], visitedLeaves: 1 });
assert.equal(exhaustiveSubsetSum([], 1).indices, null);
assert.ok(verifySubsetSum([-3, 7], 4, exhaustiveSubsetSum([-3, 7], 4).indices));
console.log('All Chapter 15 checks passed.');

## Explain without looking

Why are 2n and 3n both linear? Why does this bubble sort stay quadratic on sorted input? What extra memory does tracing binary search allocate? Why does verifying one subset not provide a fast way to find it?

Adapted from the supplied Chapter 15 Algorithm Complexity text. All seven exercises have executable worked solutions above.